<a href="https://colab.research.google.com/github/annihlj/AIOps_PromptGuard/blob/main/notebooks/train_encoder_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Guard: Milestone 2C, encoder model with LoRA (Google Colab)

**The problem.** Our TF-IDF and SVM models only see character pieces, not the meaning of a sentence. With
harmless prompts in training, the best SVM lets **15 of 60** warn prompts (names, e-mails) of our 200 German
prompts through as `allow` (LogReg: 20 of 60). The models learned the shortcut "request style = allow".

**The question.** Is a model that reads the **context** better? We fine-tune a pre-trained transformer
(an *encoder*) as a classifier for allow / warn / block.

| Part | What we use | Why |
|---|---|---|
| Base model | `xlm-roberta-base` (Hugging Face) | pre-trained on 100 languages, including de / en / fr / it |
| Head | a new classification layer with 3 outputs | allow / warn / block |
| Training method | **LoRA** (library `peft`) | trains only about 1 % of the weights: fast on a free Colab GPU, the result is only a few MB |
| Training data | about 30,000 ai4privacy train texts (balanced by class and language) **+ all harmless train prompts** | same data sources as the best SVM run (`svm_harmless`) |

**Same as before:** split (`data/splits.csv`), policy (`config/policy.yaml`), harmless prompts
(`src/harmless_data.py`), the `evaluate` function, the rule for choosing the block threshold, and the MLflow
metric names. So the numbers can be compared directly with `svm_linear.ipynb`.

**Run 1 and run 2.** One switch in chapter 3 (`USE_TEMPLATE_PII`) chooses the training data:

| Run | `USE_TEMPLATE_PII` | Training data | MLflow run |
|---|---|---|---|
| 1 | `False` | ai4privacy sample + harmless prompts | `xlmr_lora_benign` |
| 2 | `True` | the same **+ 12,000 short prompts with and without personal data** (`src/short_prompts.py`) | `xlmr_lora_benign_templates` |

Everything else is identical, so a difference between the runs comes only from the new data.

**Rules:** train only on train, choose settings on validation, never touch a test set, our 200 own prompts are
only used for evaluation.

**Structure**
0. Before you start · 1. Project and packages · 2. Setup · 3. Settings · 4. Data · 5. Training sample ·
6. Helper functions · 7. Model with LoRA · 8. Training · 9. Validation and block threshold · 10. Own prompts · 10b. Short-prompt validation ·
11. Confusion matrices · 12. Save the model · 13. Response time on the CPU · 14. Comparison with the SVM ·
15. MLflow · 16. Check MLflow and download a local copy · 17. Conclusion

## 0. Before you start (once)

1. **The code must be on GitHub.** The notebook downloads the project with `git clone` (branch `main`).
2. **GPU:** *Runtime → Change runtime type → T4 GPU*.
3. **DagsHub token as Colab secrets** (key icon on the left, then "Add new secret", switch on "Notebook access"):
   - `MLFLOW_TRACKING_USERNAME` = your DagsHub user name
   - `MLFLOW_TRACKING_PASSWORD` = your DagsHub token (DagsHub → Settings → Tokens)

   Never write the token into a cell.
4. *Runtime → Run all*. At the end the browser downloads `xlmr_lora_benign.zip` (local copy of the model).

Duration on a T4: about 30 to 45 minutes for run 1, a bit longer for run 2 (more training texts).

## 1. Project and packages

We download the project from GitHub and install the packages the training needs from `requirements.txt`
(same versions as on our laptops), **except torch**: Colab already has a torch version that matches its
GPU driver. All other packages (pandas, scikit-learn, ...) are already installed in Colab.
This cell must run **before** anything else is imported, so the new package versions are used.

When the notebook runs on a computer of its own (not in Colab), it uses the local project folder instead.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/annihlj/AIOps_PromptGuard.git"
BRANCH = "main"

if IN_COLAB:
    # fail early if the GPU runtime was forgotten
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then run again.")
    PROJECT_DIR = Path("/content/AIOps_PromptGuard")
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
    # From requirements.txt we only install what Colab does not have, with exactly our versions.
    # Not torch: Colab's own torch matches its GPU driver.
    NEEDED = {"transformers", "peft", "accelerate", "mlflow", "python-dotenv"}
    requirements = (PROJECT_DIR / "requirements.txt").read_text().splitlines()
    packages = [line.strip() for line in requirements if line.split("==")[0].strip() in NEEDED]
    print("Installing:", packages)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
else:
    PROJECT_DIR = Path("..").resolve()     # notebook started from the folder notebooks/

# The exact code version, logged to MLflow (reproducibility)
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT_DIR,
                            capture_output=True, text=True, check=False).stdout.strip()
print("Project:", PROJECT_DIR, "| git commit:", GIT_COMMIT)

## 2. Setup

Imports, GPU check, MLflow (shared DagsHub server, same experiment `promptguard` as all other runs) and
Google Drive.

In [ ]:
import gc
import json
import os
import platform
import time

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import peft
import torch
import transformers
import yaml
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, recall_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

sys.path.append(str(PROJECT_DIR / "src"))
from data_utils import load_all_splits, split_fingerprint
from harmless_data import harmless_fingerprint, load_harmless
from encoder_utils import (CLASSES, CONFIG_FILE, LORA_LAYERS, load_encoder, predict_proba,
                           predict_with_threshold)

# ---- GPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    DEVICE_NAME = f"cuda ({torch.cuda.get_device_name(0)})"
else:
    DEVICE = "cpu"
    DEVICE_NAME = f"cpu ({platform.processor() or platform.machine()})"
    print("WARNING: no GPU, training on the CPU takes many hours.")
print("Device:", DEVICE_NAME)
print(f"Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}"
      f" | peft {peft.__version__}")

# ---- MLflow: in Colab the login comes from the Colab secrets, on a laptop from .env (like the other notebooks)
if IN_COLAB:
    from google.colab import userdata
    os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/annihlj/AIOps_PromptGuard.mlflow"
    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_TRACKING_PASSWORD")
else:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_DIR / ".env")
if os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
else:
    mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_DIR / 'mlflow.db').as_posix()}")
mlflow.set_experiment("promptguard")      # also tests the login: fails here, not after the training
print("MLflow:", mlflow.get_tracking_uri())

# ---- Google Drive (only in Colab): a copy of the model is saved there at the end
# ---- Google Drive: optional backup. Turned off, the model is saved to MLflow anyway.
USE_DRIVE = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Settings

**`USE_TEMPLATE_PII`** is the only switch between run 1 (`False`) and run 2 (`True`). All other settings are
the same in both runs.

All settings in one place. They are all logged to MLflow. To try another base model later
(e.g. `microsoft/mdeberta-v3-base`), change `BASE_MODEL`, `BASE_MODEL_REVISION` and `RUN_NAME`.

**Why a fixed `BASE_MODEL_REVISION`?** Models on Hugging Face can be updated. The revision is the commit hash
of the model files, so everybody always downloads exactly the same base model.

In [ ]:
# False = run 1 (ai4privacy + harmless prompts), True = run 2 (+ Olga's short prompts, src/short_prompts.py)
USE_TEMPLATE_PII = True
RUN_NAME = "xlmr_lora_benign_templates" if USE_TEMPLATE_PII else "xlmr_lora_benign"
TRAIN_DATA = "train+benign+template_pii" if USE_TEMPLATE_PII else "train+benign"
RUN1_ID = "f71d6ccdedc449e88a91b23e73cf7b34"     # MLflow run of run 1, used for the comparison in chapter 14
BASE_MODEL = "FacebookAI/xlm-roberta-base"
BASE_MODEL_REVISION = "e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"

# LoRA: r = size of the small extra matrices, alpha = how strongly they count (alpha / r = 2),
# dropout = randomly switch off 10 % of the LoRA inputs during training (against overfitting)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1

LEARNING_RATE = 2e-4        # higher than for full fine-tuning: LoRA trains only a few weights
EPOCHS = 3                  # maximum; early stopping can end earlier
EARLY_STOPPING_PATIENCE = 1 # stop if the validation macro F1 does not improve for 1 epoch
MAX_LENGTH = 128            # tokens per text; longer texts are cut off
BATCH_SIZE = 32
N_PER_CLASS_AND_LANGUAGE = 2500   # 3 classes x 4 languages x 2,500 = about 30,000 ai4privacy texts
SEED = 42

# Expected fingerprints of the data (see README). Same data = same fingerprint.
EXPECTED_SPLIT_FINGERPRINT = "6afd6a257de1"
EXPECTED_HARMLESS_FINGERPRINT = "32d24c900fd7"
EXPECTED_TEMPLATE_FINGERPRINT = "bb97fd4d3e90"   # data/short_prompts.csv (only used in run 2)

# The same targets and the same probability thresholds as in the TF-IDF notebooks
TARGET_RECALL_BLOCK = 0.95       # at least 95 % of block texts must be blocked
TARGET_FALSE_BLOCK = 0.05        # at most 5 % of allow texts may be blocked
THRESHOLDS = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.10, 0.05]

MODEL_DIR = PROJECT_DIR / "models" / RUN_NAME                  # the trained model (not in Git)
OUT_DIR = PROJECT_DIR / "models" / f"{RUN_NAME}_outputs"       # pictures and tables for MLflow
CHECKPOINT_DIR = Path("/content/checkpoints") if IN_COLAB else PROJECT_DIR / "models" / "checkpoints"
DRIVE_DIR = Path("/content/drive/MyDrive/promptguard/models")

with open(PROJECT_DIR / "config" / "policy.yaml", encoding="utf-8") as f:
    POLICY_VERSION = str(yaml.safe_load(f)["version"])

transformers.set_seed(SEED)      # sets the seed for Python, numpy and torch at once
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print("Policy version:", POLICY_VERSION, "| run:", RUN_NAME, "| training data:", TRAIN_DATA)

## 4. Data

The raw ai4privacy files and the harmless prompts are not in Git. We create them with the **same scripts** as
on the laptop. The fingerprints prove that we get exactly the same data as everybody else.

Then the same steps as in `svm_linear.ipynb` chapter 2: one table for train and one for validation, with a column
`origin` (ai4privacy or harmless). **No test set is loaded.**

**Run 2 only:** the short prompts (`data/short_prompts.csv`) are built with Olga's script and checked with
their fingerprint. Their **train** part goes into training. Their **validation** part (other sentence frames)
is only used to evaluate in chapter 10b; it is **not** added to the validation set, so early stopping and
the block threshold use exactly the same data as in run 1. Their **test** part is not loaded (Milestone 5).

In [ ]:
data_dir = PROJECT_DIR / "data"
if not (data_dir / "german_pii_52k.jsonl").exists():
    subprocess.run([sys.executable, "src/download_data.py"], cwd=PROJECT_DIR, check=True)
if not (data_dir / "harmless_prompts.csv").exists():
    print("Building the harmless prompts (a few minutes) ...")
    # the script prints a long report (spot check); we only show the last lines
    result = subprocess.run([sys.executable, "src/harmless_data.py"], cwd=PROJECT_DIR, check=True,
                            capture_output=True, text=True, encoding="utf-8")
    print("\n".join(result.stdout.splitlines()[-2:]))

if USE_TEMPLATE_PII and not (data_dir / "short_prompts.csv").exists():
    print("Building the short prompts (a few minutes) ...")
    result = subprocess.run([sys.executable, "src/short_prompts.py"], cwd=PROJECT_DIR, check=True,
                            capture_output=True, text=True, encoding="utf-8")
    print("\n".join(result.stdout.splitlines()[-2:]))

SPLIT_FINGERPRINT = split_fingerprint()
HARMLESS_FINGERPRINT = harmless_fingerprint()
print("Split fingerprint:   ", SPLIT_FINGERPRINT)
print("Harmless fingerprint:", HARMLESS_FINGERPRINT)
assert SPLIT_FINGERPRINT == EXPECTED_SPLIT_FINGERPRINT, "splits.csv is different from the team version"
assert HARMLESS_FINGERPRINT == EXPECTED_HARMLESS_FINGERPRINT, (
    "harmless_prompts.csv is different from the team version. Fix: upload your local "
    "data/harmless_prompts.csv into the folder data/ of the project (folder icon on the left) "
    "and run this cell again.")

TEMPLATE_FINGERPRINT = None
if USE_TEMPLATE_PII:
    from short_prompts import load_short_prompts, short_prompts_fingerprint
    TEMPLATE_FINGERPRINT = short_prompts_fingerprint()
    print("Template fingerprint:", TEMPLATE_FINGERPRINT)
    assert TEMPLATE_FINGERPRINT == EXPECTED_TEMPLATE_FINGERPRINT, (
        "short_prompts.csv is different from the team version (see README). Fix: upload your local "
        "data/short_prompts.csv into the folder data/ of the project and run this cell again.")

In [ ]:
splits = load_all_splits()
pii_train, pii_val = splits["train"], splits["validation"]
del splits                                     # no test set in this notebook

harmless_train = load_harmless("train")
harmless_val = load_harmless("validation")     # we deliberately do not load the test part

COLUMNS = ["source_text", "decision", "language"]

def tag(df, origin):
    # keep only the columns we need and add where the text comes from
    return df[COLUMNS].assign(origin=origin)

train_parts = [tag(pii_train, "ai4privacy"), tag(harmless_train, "harmless")]
if USE_TEMPLATE_PII:
    # run 2: Olga's short prompts (train part only)
    train_parts.append(tag(load_short_prompts("train"), "template_pii"))
train = pd.concat(train_parts, ignore_index=True)
val = pd.concat([tag(pii_val, "ai4privacy"), tag(harmless_val, "harmless")], ignore_index=True)
own = pd.read_csv(PROJECT_DIR / "data" / "own_prompts.csv")     # our 200 German prompts, ONLY for evaluation
# run 2: short-prompt validation (other sentence frames than in training), ONLY for evaluation (chapter 10b)
template_val = tag(load_short_prompts("validation"), "template_pii") if USE_TEMPLATE_PII else None

del pii_train, pii_val
_ = gc.collect()

is_pii = (val["origin"] == "ai4privacy").to_numpy()      # validation rows from ai4privacy
is_harmless = ~is_pii                                     # validation rows from the harmless prompts
is_de = (val["language"] == "de").to_numpy()

display(pd.crosstab(val["origin"], val["decision"], margins=True).rename_axis("VALIDATION"))

## 5. Training sample

Training a transformer on all 146,000 train texts would take hours. We take a **sample**:

- from ai4privacy train: **2,500 texts per class and language** (3 × 4 groups). So every class and every
  language is equally represented. (English allow has only 2,481 texts, so it gets all of them.)
- **plus all harmless train prompts** (like the `svm_harmless` run).
- **run 2 only: plus all 12,000 short train prompts** (2,000 per class in German and English, like Olga's
  `svm_short` run). They are added completely, not sampled.

`random_state=42`, so everybody gets the same sample. The validation set stays complete.

In [ ]:
pii_train = train[train["origin"] == "ai4privacy"]
# shuffle once (seed 42), then keep the first 2,500 texts of every class x language group
pii_sample = (pii_train.sample(frac=1, random_state=SEED)
              .groupby(["decision", "language"]).head(N_PER_CLASS_AND_LANGUAGE))
sample_parts = [pii_sample, train[train["origin"] == "harmless"]]
if USE_TEMPLATE_PII:
    sample_parts.append(train[train["origin"] == "template_pii"])     # run 2: all short train prompts
train_sample = (pd.concat(sample_parts)
                .sample(frac=1, random_state=SEED)          # mix the rows of all sources
                .reset_index(drop=True))
N_HARMLESS_TRAIN = int((train_sample["origin"] == "harmless").sum())
# number of short prompts per class (logged to MLflow in run 2)
N_TEMPLATE_TRAIN = train_sample.loc[train_sample["origin"] == "template_pii", "decision"].value_counts().to_dict()
del train, pii_train, pii_sample
_ = gc.collect()

display(pd.crosstab([train_sample["origin"], train_sample["decision"]], train_sample["language"], margins=True)
        .rename_axis(["TRAIN SAMPLE", ""]))
print(f"Training texts: {len(train_sample):,} (of which harmless: {N_HARMLESS_TRAIN:,}, "
      f"short template prompts: {sum(N_TEMPLATE_TRAIN.values()):,})")
display(train_sample["decision"].value_counts().rename("training texts per class").to_frame())

## 6. Helper functions

`evaluate`, `harmless_shares`, `threshold_sweep`, `choose_threshold` and `plot_sweep` are **copied unchanged**
from `svm_linear.ipynb` chapter 3 (only the axis label of the plot says probability instead of score).
`predict_with_threshold` comes from `src/encoder_utils.py` and uses the same rule as the TF-IDF notebooks:
**block if p(block) ≥ t**, otherwise the more likely of allow / warn.

In [ ]:
def evaluate(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    is_allow, is_warn = y_true == "allow", y_true == "warn"
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "recall_block": recall_score(y_true, y_pred, labels=["block"], average="macro", zero_division=0),
        "recall_warn": recall_score(y_true, y_pred, labels=["warn"], average="macro", zero_division=0),
        "recall_allow": recall_score(y_true, y_pred, labels=["allow"], average="macro", zero_division=0),
        "false_block_allow": float((y_pred[is_allow] == "block").mean()) if is_allow.any() else np.nan,
        "false_block_warn": float((y_pred[is_warn] == "block").mean()) if is_warn.any() else np.nan,
    }


def harmless_shares(y_pred) -> dict:
    y_pred = np.asarray(y_pred)
    return {f"harmless -> {c}": float((y_pred == c).mean()) for c in CLASSES}


def threshold_sweep(scores, classes, y_true, rows_mask):
    # tries every threshold on the given rows and returns one table row per threshold
    rows = []
    for t in THRESHOLDS:
        pred = predict_with_threshold(scores[rows_mask], classes, t)
        rows.append({"threshold": t, **evaluate(np.asarray(y_true)[rows_mask], pred)})
    sweep = pd.DataFrame(rows)
    sweep["meets both targets"] = ((sweep["recall_block"] >= TARGET_RECALL_BLOCK)
                                   & (sweep["false_block_allow"] <= TARGET_FALSE_BLOCK))
    return sweep


def choose_threshold(sweep):
    # The same rule as in the TF-IDF notebooks, written down before looking at the results:
    # 1. highest threshold that reaches both targets, otherwise
    # 2. best recall block among thresholds with <= 5 % harmless texts blocked
    both = sweep[sweep["meets both targets"]]
    if len(both):
        return float(both["threshold"].max()), "highest threshold that reaches both targets"
    safe = sweep[sweep["false_block_allow"] <= TARGET_FALSE_BLOCK]
    if len(safe):
        return float(safe.loc[safe["recall_block"].idxmax(), "threshold"]), \
            "both targets not reachable; best recall block with <= 5 % false alarms"
    return None, "no threshold keeps false alarms <= 5 %; keep the default rule"


def plot_sweep(sweep, title):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sweep["threshold"], sweep["recall_block"], marker="o", color=BLUE, label="recall block (higher = safer)")
    ax.plot(sweep["threshold"], sweep["false_block_allow"], marker="o", color=ORANGE, label="allow texts blocked (lower = better)")
    ax.axhline(TARGET_RECALL_BLOCK, color=BLUE, linestyle="--", linewidth=1)
    ax.axhline(TARGET_FALSE_BLOCK, color=ORANGE, linestyle="--", linewidth=1)
    ax.invert_xaxis()
    ax.set_xlabel("block threshold t  (prompt is blocked if p(block) >= t)")
    ax.set_ylabel("share")
    ax.set_title(title)
    ax.legend(loc="center right")
    plt.tight_layout()
    plt.show()

## 7. Model with LoRA

**Tokenizer.** A transformer does not read letters but *tokens* (word pieces, e.g. "Urlaubs", "regel", "ung").
The tokenizer turns every text into a list of token numbers, at most `MAX_LENGTH` = 128.

**Model.** `AutoModelForSequenceClassification` = the pre-trained xlm-roberta-base plus a **new classification
head** with 3 outputs (allow, warn, block). The head starts with random weights, so it must be trained.

**LoRA (Low-Rank Adaptation).** Instead of changing all ~280 million weights of the model, LoRA **freezes** them
and adds small extra matrices (rank `r` = 16) to the attention layers `query` and `value`. Only these small
matrices and the classification head are trained. Advantages: much faster, fits on a free GPU, and the result
(the *adapter*) is only a few MB instead of about 1 GB.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
label2id = {c: i for i, c in enumerate(CLASSES)}      # allow = 0, warn = 1, block = 2


class PromptDataset(torch.utils.data.Dataset):
    """Texts -> token numbers (done once at the start) + the label as a number."""

    def __init__(self, texts, decisions):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.labels = [label2id[d] for d in decisions]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        # one training example: token numbers, attention mask and the label
        item = {key: values[i] for key, values in self.encodings.items()}
        item["labels"] = self.labels[i]
        return item


train_ds = PromptDataset(train_sample["source_text"], train_sample["decision"])
val_ds = PromptDataset(val["source_text"], val["decision"])
cut_off = np.mean([len(ids) >= MAX_LENGTH for ids in train_ds.encodings["input_ids"]])
print(f"Train texts longer than {MAX_LENGTH} tokens (cut off): {cut_off:.1%}")

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION, num_labels=len(CLASSES),
    id2label=dict(enumerate(CLASSES)), label2id=label2id)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,       # sequence classification = one label per text
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    **LORA_LAYERS[BASE_MODEL],        # target_modules (query, value) + modules_to_save (classifier head)
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
N_TRAINABLE = sum(p.numel() for p in model.parameters() if p.requires_grad)

## 8. Training

The Hugging Face `Trainer` does the training loop for us. Important settings:

- After **every epoch** (one pass through all training texts) it measures the **macro F1 on the validation set**.
- **Early stopping:** if the macro F1 does not improve for one epoch, training stops.
- `load_best_model_at_end`: at the end we keep the epoch with the **best** validation macro F1, not the last one.
- `fp16`: calculates in half precision on the GPU, about twice as fast, same quality.
- MLflow logging by the Trainer is switched off (`report_to="none"`); we log everything ourselves in chapter 15,
  with the same names as in the other runs.

In [ ]:
def compute_metrics(eval_pred):
    # called by the Trainer after every epoch on the validation set (rule: highest probability wins)
    logits, labels = eval_pred
    return {"macro_f1": f1_score(labels, logits.argmax(axis=1), average="macro")}


args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=128,
    warmup_steps=0.06,                 # a number < 1 = share of all steps: small learning rate for the first 6 %
    weight_decay=0.01,
    eval_strategy="epoch",             # validate after every epoch ...
    save_strategy="epoch",             # ... and save a checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=(DEVICE == "cuda"),
    logging_steps=100,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),   # fills texts of one batch up to the same length
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

start = time.perf_counter()
trainer.train()
TRAIN_MINUTES = (time.perf_counter() - start) / 60
EPOCHS_TRAINED = float(trainer.state.epoch)

# validation macro F1 after every epoch
history = pd.DataFrame([h for h in trainer.state.log_history if "eval_macro_f1" in h])
history = history[["epoch", "eval_macro_f1", "eval_loss"]].round(4)
display(history)
print(f"Training took {TRAIN_MINUTES:.0f} min, {EPOCHS_TRAINED:g} epochs. "
      f"Best epoch: {history.loc[history['eval_macro_f1'].idxmax(), 'epoch']:g}")

## 9. Validation and block threshold

Like `svm_linear.ipynb` part B: we calculate the probabilities for the **combined** validation set
(ai4privacy + real harmless prompts) once, try all thresholds and choose one with the same rule as before.
For comparison we also show the default rule (highest probability wins, "argmax").

In [ ]:
model.eval()
val_proba = predict_proba(model, tokenizer, val["source_text"].tolist(), MAX_LENGTH, batch_size=128)

all_rows = np.ones(len(val), dtype=bool)
sweep = threshold_sweep(val_proba, CLASSES, val["decision"], all_rows)
display(sweep.round(4))
plot_sweep(sweep, "Encoder: threshold trade-off (combined validation)")

BLOCK_THRESHOLD, threshold_reason = choose_threshold(sweep)
val_pred = predict_with_threshold(val_proba, CLASSES, BLOCK_THRESHOLD)
val_pred_argmax = predict_with_threshold(val_proba, CLASSES, None)


def val_metrics(pred):
    # the same metrics as the svm_harmless run (combined validation)
    m = evaluate(val["decision"], pred)
    m["harmless_to_block"] = float((pred[is_harmless] == "block").mean())
    m["harmless_to_warn"] = float((pred[is_harmless] == "warn").mean())
    m["de_recall_block"] = recall_score(val["decision"][is_de], pred[is_de], labels=["block"], average="macro")
    return m


metrics_val = val_metrics(val_pred)
metrics_val_argmax = val_metrics(val_pred_argmax)
print(f"Chosen threshold: {BLOCK_THRESHOLD} ({threshold_reason})")
display(pd.DataFrame({f"threshold {BLOCK_THRESHOLD}": metrics_val, "argmax": metrics_val_argmax}).round(4))

## 10. Our German prompts (evaluation only)

`data/own_prompts.csv` has two versions (column `version`, `src/own_prompts.py`):

| Version | Prompts | allow / warn / block |
|---|---|---|
| v1 | the original 200; **all earlier runs were measured on these** | 80 / 60 / 60 |
| v2 | 400 more, added later | 120 / 140 / 140 |
| all | 600 | 200 / 200 / 200 |

We measure all three. `own_prompts_*` in MLflow and the comparison table in chapter 14 stay on **v1**, so they can
be compared with the earlier runs. In addition `own_prompts_v1_*`, `own_prompts_v2_*` and `own_prompts_all_*` are
logged: these names mean the same in every run.

The most important question: how many **warn prompts** (names, e-mails, ...) does the model let through as `allow`?

In [ ]:
own_proba = predict_proba(model, tokenizer, own["prompt"].tolist(), MAX_LENGTH)
own_pred_all = predict_with_threshold(own_proba, CLASSES, BLOCK_THRESHOLD)
expected_all = own["expected_decision"].to_numpy()
casual_all = (own["style"] == "casual").to_numpy()
# older files have no column "version": then all prompts are v1
own_version = own["version"].to_numpy() if "version" in own.columns else np.full(len(own), "v1")


def own_metrics(mask):
    # the own_prompts_* metrics (same definitions as in all earlier runs) for some of the prompts
    e, p, c = expected_all[mask], own_pred_all[mask], casual_all[mask]
    m = evaluate(e, p)
    m["accuracy"] = float((p == e).mean())
    m["warn_to_allow"] = float((p[e == "warn"] == "allow").mean())
    m["casual_recall_block"] = float((p[(e == "block") & c] == "block").mean())
    return m


own_subsets = {"v1": own_version == "v1", "v2": own_version == "v2", "all": np.ones(len(own), dtype=bool)}
own_subsets = {name: mask for name, mask in own_subsets.items() if mask.any()}
metrics_own_by_version = {name: own_metrics(mask) for name, mask in own_subsets.items()}
metrics_own = metrics_own_by_version["v1"]          # = own_prompts_* in MLflow, comparable with runs 1 and 2
display(pd.DataFrame(metrics_own_by_version).round(3))

# From here on (confusion matrix, comparison table) only v1, so the numbers stay comparable
is_v1 = own_subsets["v1"]
own_pred, expected, casual = own_pred_all[is_v1], expected_all[is_v1], casual_all[is_v1]
display(pd.crosstab(pd.Series(expected, name="expected (v1)"), pd.Series(own_pred, name="predicted"))
        .reindex(index=CLASSES, columns=CLASSES, fill_value=0))
for name, mask in own_subsets.items():
    warn = mask & (expected_all == "warn")
    print(f"{name}: warn prompts let through (allow): {(own_pred_all[warn] == 'allow').sum()} of {warn.sum()}")

# all wrongly classified own prompts (all versions), with the probabilities (for the team discussion)
wrong = own.assign(predicted=own_pred_all, p_allow=own_proba[:, 0].round(3), p_warn=own_proba[:, 1].round(3),
                   p_block=own_proba[:, 2].round(3))
wrong = wrong[wrong["predicted"] != wrong["expected_decision"]]
OUT_DIR.mkdir(parents=True, exist_ok=True)
wrong.to_csv(OUT_DIR / "own_prompts_wrong.csv", index=False)
show = [c for c in ["version", "prompt", "expected_decision", "predicted", "p_allow", "p_warn", "p_block",
                    "category", "style"] if c in wrong.columns]
with pd.option_context("display.max_colwidth", 100, "display.max_rows", 300):
    display(wrong[show])

## 10b. Short-prompt validation (run 2 only)

Olga's validation prompts: 600 short requests (100 per class in German and English) built from **other sentence
frames** than the training prompts. A good result here means the model learned the **content** (is there a name,
an IBAN?), not the frames. The four numbers have the same definitions as the rows "short val: ..." in
`reports/short_prompts_before_after.csv`, so they can be compared with Olga's SVM.

In [ ]:
def template_rows(pred, y):
    # the same four rows as in short_prompts_retrain.ipynb ("short val: ...")
    return {
        "short val: correct": f"{(pred == y).mean():.1%}",
        "short val: block caught": f"{(pred[y == 'block'] == 'block').mean():.1%}",
        "short val: warn let through (allow)": f"{(pred[y == 'warn'] == 'allow').mean():.1%}",
        "short val: allow blocked": f"{(pred[y == 'allow'] == 'block').mean():.1%}",
    }


metrics_template = {}
if USE_TEMPLATE_PII:
    template_y = template_val["decision"].to_numpy()
    template_proba = predict_proba(model, tokenizer, template_val["source_text"].tolist(), MAX_LENGTH, batch_size=128)
    template_pred = predict_with_threshold(template_proba, CLASSES, BLOCK_THRESHOLD)
    metrics_template = evaluate(template_y, template_pred)
    metrics_template["accuracy"] = float((template_pred == template_y).mean())
    metrics_template["warn_to_allow"] = float((template_pred[template_y == "warn"] == "allow").mean())
    display(pd.Series(template_rows(template_pred, template_y)).to_frame("short-prompt validation"))
    display(pd.crosstab(pd.Series(template_y, name="expected"), pd.Series(template_pred, name="predicted"))
            .reindex(index=CLASSES, columns=CLASSES, fill_value=0))
else:
    print("Run 1: skipped (USE_TEMPLATE_PII = False)")

## 11. Confusion matrices

Rows = correct class, columns = predicted class. The diagonal is correct. The most dangerous cells are
*block → allow* and *warn → allow* (data leaves the company without a warning).

In [ ]:
fig, axes = plt.subplots(1, 3 if USE_TEMPLATE_PII else 2, figsize=(16 if USE_TEMPLATE_PII else 11, 4.5))
ConfusionMatrixDisplay.from_predictions(val["decision"], val_pred, labels=CLASSES, ax=axes[0],
                                        cmap="Blues", colorbar=False, values_format=",d")
axes[0].set_title(f"Validation ({len(val):,} texts)")
ConfusionMatrixDisplay.from_predictions(expected, own_pred, labels=CLASSES, ax=axes[1],
                                        cmap="Blues", colorbar=False)
axes[1].set_title(f"Our German prompts (v1, {len(expected)})")
if USE_TEMPLATE_PII:
    ConfusionMatrixDisplay.from_predictions(template_y, template_pred, labels=CLASSES, ax=axes[2],
                                            cmap="Blues", colorbar=False)
    axes[2].set_title(f"Short-prompt validation ({len(template_val)})")
for ax in axes:
    ax.set_xlabel("predicted")
    ax.set_ylabel("correct")
fig.suptitle(f"{RUN_NAME}, block threshold {BLOCK_THRESHOLD}")
plt.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

## 12. Save the model

We save the **LoRA adapter** (trained LoRA weights + classification head), the tokenizer and a small file
`promptguard_config.json` with everything needed to use the model later (base model + revision, threshold, ...).
The base model itself is not saved: it is downloaded from Hugging Face at the fixed revision.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
model_config = {
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "classes": CLASSES,
    "max_length": MAX_LENGTH,
    "block_threshold": BLOCK_THRESHOLD,
    "split_fingerprint": SPLIT_FINGERPRINT,
    "harmless_fingerprint": HARMLESS_FINGERPRINT,
    "policy_version": POLICY_VERSION,
    "git_commit": GIT_COMMIT,
}
(MODEL_DIR / CONFIG_FILE).write_text(json.dumps(model_config, indent=2), encoding="utf-8")
size_mb = sum(f.stat().st_size for f in MODEL_DIR.iterdir()) / 1e6
print(f"Saved to {MODEL_DIR} ({size_mb:.1f} MB):", sorted(f.name for f in MODEL_DIR.iterdir()))

## 13. Response time on the CPU

PromptGuard will run on a server without GPU, so we measure on the **CPU**, **one prompt at a time**, with the
same 1,000 validation prompts as in `svm_linear.ipynb` chapter 9. We load the **saved** model for this
(exactly like `src/measure_latency.py` does), which also checks that saving worked: the reloaded model must
give the same decisions on our own prompts.

**Only a guide value:** the Colab CPU is slow and shared with other users. The fair comparison with TF-IDF / SVM
(3.5 ms on the laptop) comes from `python src/measure_latency.py --download --log` on the same laptop.

In [ ]:
cpu_model, cpu_tokenizer, cpu_cfg = load_encoder(MODEL_DIR, device="cpu", merge=True)

# check: the saved model makes the same decisions as the model in memory
reloaded_pred = predict_with_threshold(predict_proba(cpu_model, cpu_tokenizer, own["prompt"].tolist(), MAX_LENGTH),
                                       CLASSES, cpu_cfg["block_threshold"])
print(f"Reloaded model: {(reloaded_pred == own_pred_all).sum()} of {len(own)} own prompts with the same decision")

latency_sample = val["source_text"].sample(1000, random_state=SEED).tolist()   # same sample as svm_linear.ipynb


def classify(prompt):
    proba = predict_proba(cpu_model, cpu_tokenizer, [prompt], MAX_LENGTH, batch_size=1)
    return predict_with_threshold(proba, CLASSES, BLOCK_THRESHOLD)[0]


for prompt in latency_sample[:20]:     # warm-up, not counted
    classify(prompt)
times = []
for prompt in latency_sample:
    start = time.perf_counter()
    classify(prompt)
    times.append((time.perf_counter() - start) * 1000)
COLAB_P50, COLAB_P95 = float(np.percentile(times, 50)), float(np.percentile(times, 95))
print(f"Colab CPU ({torch.get_num_threads()} threads): p50 = {COLAB_P50:.1f} ms, p95 = {COLAB_P95:.1f} ms")
del cpu_model
_ = gc.collect()

## 14. Comparison

Run 1 (`USE_TEMPLATE_PII = False`): SVM + harmless vs. this encoder.
Run 2 (`USE_TEMPLATE_PII = True`): four columns, so that SVM and encoder can be compared on the **same training data**:

| Column | Where the numbers come from |
|---|---|
| SVM + harmless | `reports/model_comparison_m2.csv` (`svm_linear.ipynb`) |
| SVM + short prompts | `reports/short_prompts_before_after.csv` (`short_prompts_retrain.ipynb`) |
| Encoder run 1 | the table `model_comparison_encoder.csv` saved in the MLflow run of run 1. Run 1 was never evaluated on the short prompts, so these four rows are calculated here with the **saved run 1 model** from MLflow (nothing is estimated, run 1 itself is not changed). |
| Encoder run 2 | this run |

The "own ..." rows are measured on **v1** (the 200 prompts all columns were measured on). The results on v2 and
all 600 prompts are in MLflow (`own_prompts_v2_*`, `own_prompts_all_*`) and in
`reports/own_prompts_v2_comparison.csv` (`python src/evaluate_own_prompts.py`).
Rows a model was not measured on show "n/a". Response times are **not** comparable between the columns
(laptop vs. Colab CPU); the fair laptop value comes from `python src/measure_latency.py`.

In [ ]:
def count(mask_pred, mask_total):
    return f"{int(mask_pred.sum())} of {int(mask_total.sum())}"


pii = evaluate(val["decision"][is_pii], val_pred[is_pii])
h = harmless_shares(val_pred[is_harmless])
own_block = expected == "block"
encoder_column = {
    "block threshold": BLOCK_THRESHOLD,
    "ai4privacy recall block": round(pii["recall_block"], 3),
    "ai4privacy recall block (de)": round(recall_score(val["decision"][is_pii & is_de], val_pred[is_pii & is_de],
                                                       labels=["block"], average="macro"), 3),
    "ai4privacy macro F1": round(pii["macro_f1"], 3),
    "ai4privacy allow texts blocked": round(pii["false_block_allow"], 3),
    "real harmless prompts allowed": f"{h['harmless -> allow']:.1%}",
    "real harmless prompts blocked": f"{h['harmless -> block']:.1%}",
    "own prompts correct": count(own_pred == expected, expected == expected),
    "own: block prompts caught": count(own_pred[own_block] == "block", own_block),
    "own: casual block prompts caught": count(own_pred[own_block & casual] == "block", own_block & casual),
    "own: harmless prompts blocked": count(own_pred[expected == "allow"] == "block", expected == "allow"),
    "own: warn prompts let through (allow)": count(own_pred[expected == "warn"] == "allow", expected == "warn"),
    "response time p95 (ms)": f"{COLAB_P95:.1f} (Colab CPU)",
}
reports_dir = PROJECT_DIR / "reports"
svm = pd.read_csv(reports_dir / "model_comparison_m2.csv", index_col=0)["SVM + harmless"].astype(str)
svm["response time p95 (ms)"] += " (laptop)"

if not USE_TEMPLATE_PII:
    # run 1: the original two-column table
    comparison = pd.DataFrame({"SVM + harmless": svm, "Encoder (XLM-R + LoRA) + harmless": encoder_column})
else:
    encoder_column.update(template_rows(template_pred, template_y))

    # Olga's SVM with the same short prompts
    svm_short = pd.read_csv(reports_dir / "short_prompts_before_after.csv", index_col=0)[
        "SVM + short prompts (after)"].astype(str)
    svm_short["response time p95 (ms)"] += " (laptop)"

    # Encoder run 1: its comparison table and its model, both from MLflow
    # (downloaded next to the model folder, not into OUT_DIR, so it does not end up in the ZIP of chapter 16)
    run1_dir = Path(mlflow.artifacts.download_artifacts(run_id=RUN1_ID,
                                                        dst_path=str(MODEL_DIR.parent / "run1_from_mlflow")))
    run1 = pd.read_csv(run1_dir / "model_comparison_encoder.csv", index_col=0)[
        "Encoder (XLM-R + LoRA) + harmless"].astype(str)
    run1_model, run1_tokenizer, run1_cfg = load_encoder(run1_dir / "model", device=DEVICE, merge=True)
    run1_pred = predict_with_threshold(
        predict_proba(run1_model, run1_tokenizer, template_val["source_text"].tolist(), run1_cfg["max_length"]),
        CLASSES, run1_cfg["block_threshold"])
    run1 = pd.concat([run1, pd.Series(template_rows(run1_pred, template_y))])
    del run1_model
    _ = gc.collect()

    comparison = pd.DataFrame({
        "SVM + harmless": svm,
        "SVM + short prompts": svm_short,
        "Encoder run 1 (+ harmless)": run1,
        "Encoder run 2 (+ harmless + short prompts)": pd.Series(encoder_column).astype(str),
    })
    # same row order as before, the short-prompt rows after the ai4privacy / harmless rows
    order = list(encoder_column)[:7] + list(template_rows(template_pred, template_y)) + list(encoder_column)[7:13]
    comparison = comparison.reindex(order).fillna("n/a")

comparison.to_csv(OUT_DIR / "model_comparison_encoder.csv")
with pd.option_context("display.max_colwidth", 40):
    display(comparison)

## 15. MLflow

One run (`xlmr_lora_benign` or `xlmr_lora_benign_templates`, see `RUN_NAME`) in the experiment `promptguard`, with the **same metric names** as the SVM run
`svm_harmless` (`val_...` on the combined validation set, `own_prompts_...` on our 200 prompts).
Artifacts: the model folder (`model/`), the confusion matrix, the wrongly classified own prompts, the threshold
table and the comparison table.

**Get the model back later:** DagsHub → Experiments → run → Artifacts → `model`, or
`python src/measure_latency.py --download` (downloads it to `models/xlmr_lora_benign/`).

In [ ]:
sweep.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)

params = {
    "model": "encoder_lora",
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": ",".join(LORA_LAYERS[BASE_MODEL]["target_modules"]),
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "train_size": len(train_sample),
    "n_per_class_and_language": N_PER_CLASS_AND_LANGUAGE,
    "n_harmless_train": N_HARMLESS_TRAIN,
    "train_data": TRAIN_DATA,
    "device": DEVICE_NAME,
    "seed": SEED,
    "block_threshold": "argmax" if BLOCK_THRESHOLD is None else BLOCK_THRESHOLD,
    "threshold_reason": threshold_reason,
    "split_fingerprint": SPLIT_FINGERPRINT,
    "harmless_fingerprint": HARMLESS_FINGERPRINT,
    "policy_version": POLICY_VERSION,
    "git_commit": GIT_COMMIT,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "peft_version": peft.__version__,
}
if USE_TEMPLATE_PII:
    # run 2: the short prompts (same parameter names as Olga's svm_short run, plus the count per class)
    params.update({
        "short_prompts_fingerprint": TEMPLATE_FINGERPRINT,
        "n_short_train": sum(N_TEMPLATE_TRAIN.values()),
        **{f"n_short_train_{c}": N_TEMPLATE_TRAIN.get(c, 0) for c in CLASSES},
        "compared_with_run_id": RUN1_ID,
    })

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.set_tags({"milestone": "M2", "experiment_part": "C_encoder_lora", "candidate": "true"})
    mlflow.log_params(params)
    mlflow.log_metrics({f"val_{k}": v for k, v in metrics_val.items()})
    mlflow.log_metrics({f"val_argmax_{k}": v for k, v in metrics_val_argmax.items()})
    mlflow.log_metrics({f"own_prompts_{k}": v for k, v in metrics_own.items()})        # v1, as in runs 1 and 2
    # explicit names, the same meaning in every run: own_prompts_v1_*, own_prompts_v2_*, own_prompts_all_*
    mlflow.log_metrics({f"own_prompts_{name}_{k}": v
                        for name, m in metrics_own_by_version.items() for k, v in m.items()})
    mlflow.log_metrics({f"short_val_{k}": v for k, v in metrics_template.items()})   # empty in run 1
    mlflow.log_metrics({
        "colab_cpu_latency_p50_ms": COLAB_P50,   # laptop values come from src/measure_latency.py --log
        "colab_cpu_latency_p95_ms": COLAB_P95,
        "epochs_trained": EPOCHS_TRAINED,
        "train_minutes": TRAIN_MINUTES,
        "trainable_params": N_TRAINABLE,
        "train_cut_off_share": float(cut_off),
    })
    for _, row in history.iterrows():            # learning curve: validation macro F1 per epoch
        mlflow.log_metric("epoch_val_macro_f1", row["eval_macro_f1"], step=int(round(row["epoch"])))
    mlflow.log_artifacts(str(MODEL_DIR), artifact_path="model")
    for name in ["confusion_matrix.png", "own_prompts_wrong.csv", "threshold_sweep.csv",
                 "model_comparison_encoder.csv"]:
        mlflow.log_artifact(str(OUT_DIR / name))
    RUN_ID = run.info.run_id
print("Logged to MLflow, run id:", RUN_ID)

## 16. Check MLflow and download a local copy

**1. Check.** We ask MLflow (DagsHub) which files the run really has under `model/`, download the adapter again and
compare it byte by byte (SHA-256 hash) with the local file. If anything is missing or different, the cell stops
with an error: then run chapter 15 again.

**2. Local copy.** Model folder and result tables are packed into one ZIP file and downloaded by the browser.
Unpack it on the laptop into the folder `models/` of the project, then `python src/measure_latency.py --log`
finds the model in `models/xlmr_lora_benign/` without downloading it again.
If the browser does not start the download: folder icon on the left → `/content/xlmr_lora_benign.zip` →
three dots → *Download*. Colab deletes `/content` when the session ends, so do it before closing.

**3. Google Drive** (optional, `USE_DRIVE` in chapter 2): an extra copy.

In [ ]:
import hashlib
import tempfile
import zipfile


def sha256(path):
    # "fingerprint" of a file: same content = same hash
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# ---- 1) Did the model really arrive in MLflow?
client = mlflow.MlflowClient()
remote_files = sorted(Path(f.path).name for f in client.list_artifacts(RUN_ID, "model"))
local_files = sorted(f.name for f in MODEL_DIR.iterdir())
print("In MLflow under model/:", remote_files)
missing = sorted(set(local_files) - set(remote_files))
assert not missing, f"Missing in MLflow: {missing}. Run chapter 15 again."

with tempfile.TemporaryDirectory() as tmp:
    downloaded = mlflow.artifacts.download_artifacts(
        run_id=RUN_ID, artifact_path="model/adapter_model.safetensors", dst_path=tmp)
    assert sha256(downloaded) == sha256(MODEL_DIR / "adapter_model.safetensors"), \
        "The adapter in MLflow is different from the local one. Run chapter 15 again."
print(f"OK: all {len(local_files)} model files are in MLflow, the adapter is identical (run id {RUN_ID})")

# ---- 2) Local copy: one ZIP with the model folder and the result tables
zip_path = (Path("/content") if IN_COLAB else MODEL_DIR.parent) / f"{RUN_NAME}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [MODEL_DIR, OUT_DIR]:
        for f in sorted(folder.rglob("*")):
            # path inside the ZIP: xlmr_lora_benign/... and xlmr_lora_benign_outputs/...
            z.write(f, f.relative_to(MODEL_DIR.parent))
print(f"ZIP: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))      # the browser downloads the file

# ---- 3) Optional: copy to Google Drive
if IN_COLAB and USE_DRIVE:
    shutil.copytree(MODEL_DIR, DRIVE_DIR / RUN_NAME, dirs_exist_ok=True)
    shutil.copytree(OUT_DIR, DRIVE_DIR / f"{RUN_NAME}_outputs", dirs_exist_ok=True)
    print("Copied to", DRIVE_DIR)

## 17. Conclusion

*To be written after the run, from the table in chapter 14 and the laptop response time
(`python src/measure_latency.py --download --log`).*

**Limitations (known before the run)**
- Trained on a sample of about 34,000 texts, not on all 150,000 (TF-IDF / SVM used all of them).
- The validation set is used twice: for early stopping and for the block threshold (the same as the threshold in
  the SVM notebooks). The final, unbiased check is the test set in Milestone 5.
- Our own prompts are only 200, so one wrong prompt moves a share by 0.5 to 1.7 points.
- Run 2: the short prompts are German and English only. Their personal data values come from ai4privacy
  **train**, but ai4privacy itself repeats some values across its splits: about 3 to 11 % of the short train
  prompts with an identifier contain a value that also appears in ai4privacy validation / test. That is the
  same rate as in the ai4privacy train texts of run 1 (8.5 %), so it is a property of the dataset.
- Run 2: the short-prompt validation has one duplicate (same text, only upper / lower case differs).
- GPU training is not 100 % bit-identical between runs. With the same seed and data, results differ at most
  slightly; the saved adapter in MLflow is the reference.